# Global Geary's C: an interactive lab

In this lab, we will measure how different neighbouring values are. We will build a tiny spatial weights matrix, calculate Geary's C by hand and in Python, compare it with Moran's I, and inspect all 24 permutations of a four-value example.

**You need:** Python, NumPy and Matplotlib.

The statistic is:

$$
C = rac{n-1}{2S_0}
rac{\sum_i\sum_j w_{ij}(x_i-x_j)^2}
{\sum_i(x_i-\bar{x})^2}
$$

The weights matrix stays fixed while we rearrange the observed values.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import permutations


## 1. Set up four connected areas

The four areas form a chain. Every connection is represented in both directions in the symmetric weights matrix. This means the double sum counts each undirected edge twice.

In [ ]:
W = np.array([
    [0, 1, 0, 0],
    [1, 0, 1, 0],
    [0, 1, 0, 1],
    [0, 0, 1, 0],
], dtype=float)

values = np.array([1, 2, 3, 4], dtype=float)

def gearys_c(x, w):
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)
    n = len(x)
    mean = x.mean()
    s0 = w.sum()
    numerator = sum(
        w[i, j] * (x[i] - x[j]) ** 2
        for i in range(n)
        for j in range(n)
    )
    denominator = ((x - mean) ** 2).sum()
    return ((n - 1) / (2 * s0)) * (numerator / denominator)

def morans_i(x, w):
    x = np.asarray(x, dtype=float)
    z = x - x.mean()
    return (len(x) / w.sum()) * ((z @ w @ z) / (z @ z))

print("Geary's C:", gearys_c(values, W))
print("Moran's I:", morans_i(values, W))


In [ ]:
positions = np.arange(len(values))
fig, ax = plt.subplots(figsize=(9, 2.8))

for i in range(len(values) - 1):
    ax.plot([positions[i], positions[i + 1]], [0, 0], linewidth=3)

ax.scatter(positions, np.zeros_like(positions), s=1050, zorder=3)
for i, value in enumerate(values):
    ax.text(
        positions[i], 0, f"Area {i + 1}\nvalue = {value:g}",
        ha="center", va="center", color="white", weight="bold"
    )

ax.set_xlim(-0.5, 3.5)
ax.set_ylim(-0.7, 0.7)
ax.set_xticks([])
ax.set_yticks([])
ax.set_title("Neighbour relationships and observed values")
ax.axis("off")
plt.show()


## 2. Follow the squared differences

For the sequence `[1, 2, 3, 4]`, every neighbouring pair differs by one. Each directed edge contributes one to the numerator.

The denominator measures the overall variation around the mean. We can inspect both parts separately.

In [ ]:
mean = values.mean()
s0 = W.sum()
numerator = sum(
    W[i, j] * (values[i] - values[j]) ** 2
    for i in range(len(values))
    for j in range(len(values))
)
denominator = ((values - mean) ** 2).sum()

print("Mean:", mean)
print("S0:", s0)
print("Weighted squared-difference sum:", numerator)
print("Overall sum of squared deviations:", denominator)
print("C = (n - 1) / (2 S0) × numerator / denominator")
print("C =", (len(values) - 1) / (2 * s0) * numerator / denominator)


## 3. Change the pattern, not the geography

Now alternate high and low values. The graph remains exactly the same, but every connected pair differs by three. Squaring the difference makes each pair contribute nine times as much as before.

In [ ]:
alternating = np.array([1, 4, 1, 4], dtype=float)

print("Similar sequence, C:", gearys_c(values, W))
print("Alternating sequence, C:", gearys_c(alternating, W))
print("Similar sequence, Moran's I:", morans_i(values, W))
print("Alternating sequence, Moran's I:", morans_i(alternating, W))


## 4. Compare all possible arrangements

There are only $4! = 24$ ways to assign these four observed values to the four fixed areas. We can calculate Geary's C for every arrangement and inspect the exact permutation distribution.

For a larger dataset, enumerating every permutation quickly becomes impossible. Researchers use Monte Carlo permutations instead.

In [ ]:
all_orders = list(permutations(values))
simulated_c = np.array([gearys_c(order, W) for order in all_orders])
observed_c = gearys_c(values, W)

lower_tail = np.mean(simulated_c <= observed_c)
upper_tail = np.mean(simulated_c >= observed_c)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(simulated_c, bins=10)
ax.axvline(observed_c, linewidth=2, label=f"Observed C = {observed_c:.3f}")
ax.axvline(1, linestyle="--", linewidth=2, label="Null expectation = 1")
ax.set_xlabel("Geary's C across all arrangements")
ax.set_ylabel("Number of arrangements")
ax.set_title("Exact permutation reference distribution")
ax.legend()
plt.show()

print("Number of assignments:", len(simulated_c))
print("Mean of permutation distribution:", simulated_c.mean())
print("Exact lower-tail proportion:", lower_tail)
print("Exact upper-tail proportion:", upper_tail)


These are exact tail proportions for this tiny example because all 24 assignments were enumerated. They are not the Monte Carlo pseudo-$p$-value used when only a sample of permutations is drawn.

## What to take away

- Geary's C compares values directly across neighbouring pairs.
- Squared differences make sharp local contrasts influential.
- The null expectation is one, but significance still requires a reference distribution.
- The weights matrix defines which pairs contribute to the statistic.
- Moran's I and Geary's C offer complementary views of spatial association.

Continue with the [Geary's C lesson](../algorithms/statistics/gearys-c.md), the [Moran's I lesson](../algorithms/statistics/global-morans-i.md) and the [tested Python implementation](../implementations/python/geary-c/README.md).

**Reference:** [PySAL's official Global Geary's C guide](https://pysal.org/esda/stable/user-guide/geary.html).